# SHL Grammar Scoring: Feature Extraction

This notebook turns each audio clip into features. The modelling is done in a separate notebook that reads these files.

For every clip we extract:
- **Audio embeddings** from WavLM-large and the Whisper-large-v3 encoder (all layers, mean and std over time)
- **Transcripts** from Whisper (main) and wav2vec2 (letter by letter, no language model)
- **Text embeddings** of the transcript from RoBERTa-large and Qwen2.5-1.5B (all layers)
- **Handcrafted features**: syntax, grammar errors, language-model surprisal, speech timing

Run with GPU T4 and internet on. Takes about 1 hour (about 2.5 hours if the Whisper transcripts are not reused).

In [1]:
import os, re, gc, math, shutil, subprocess, sys, difflib, warnings
from collections import Counter

import numpy as np
import pandas as pd
import librosa
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
OUT_DIR = "/kaggle/working"
SR = 16000
DEVICE = "cuda"

# Whisper transcription is the slowest step (~1.5 h). Point this to a clips.csv from an
# earlier run to reuse its transcripts, or set it to None to transcribe everything here.
TRANSCRIPTS_CSV = "/kaggle/input/datasets/codewiz19/output-f/clips.csv"


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

## 1. Load the data

In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv").assign(split="train")
test = pd.read_csv(f"{DATA_DIR}/test.csv").assign(split="test", label=np.nan)
clips = pd.concat([train, test], ignore_index=True)[["split", "filename", "label"]]

# train and test reuse the same filenames, so the path needs the split folder
paths = DATA_DIR + "/" + clips["split"] + "/" + clips["filename"]
waves = [librosa.load(p, sr=SR)[0] for p in tqdm(paths, desc="loading audio")]

# 37 training clips are labelled 0: they are pure noise and are skipped by the model
scored = clips["label"].ne(0).to_numpy()
print(clips["split"].value_counts().to_dict())

loading audio:   0%|          | 0/985 [00:00<?, ?it/s]

{'train': 769, 'test': 216}


## 2. Audio statistics and speech timing

The loudness range separates real speech (which has pauses) from the noise-only clips. Timing features measure how much of the clip is speech and how often the speaker pauses.

In [3]:
def loudness_range_db(y, frame=400):
    n = len(y) // frame * frame
    db = 20 * np.log10(np.sqrt((y[:n].reshape(-1, frame) ** 2).mean(axis=1)) + 1e-9)
    return np.percentile(db, 95) - np.percentile(db, 10)


def timing_features(y):
    duration = len(y) / SR
    runs = librosa.effects.split(y, top_db=30, frame_length=1024, hop_length=256)
    speech = (runs[:, 1] - runs[:, 0]).sum() / SR if len(runs) else 0.0
    gaps = (runs[1:, 0] - runs[:-1, 1]) / SR if len(runs) > 1 else np.array([])
    pauses = gaps[gaps >= 0.25]
    return {
        "speech_s": speech,
        "speech_ratio": speech / duration,
        "pauses_per_min": len(pauses) / duration * 60,
        "mean_pause_s": pauses.mean() if len(pauses) else 0.0,
        "long_pauses_per_min": (pauses >= 1.0).sum() / duration * 60,
        "mean_run_s": speech / max(len(runs), 1),
    }


clips["duration_s"] = [len(y) / SR for y in waves]
clips["dyn_range_db"] = [loudness_range_db(y) for y in waves]
timing = pd.DataFrame([timing_features(y) for y in tqdm(waves, desc="timing")])

timing:   0%|          | 0/985 [00:00<?, ?it/s]

## 3. wav2vec2 transcript

wav2vec2 spells out what it hears without a language model, so it does not correct the speaker's grammar the way Whisper sometimes does. The output is messy, but how messy it is says something about the speech.

In [4]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

w2v_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h-lv60-self")
w2v_model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-960h-lv60-self").to(DEVICE).eval()


@torch.no_grad()
def w2v_transcribe(y):
    x = w2v_processor(y, sampling_rate=SR, return_tensors="pt").input_values.to(DEVICE)
    with torch.autocast("cuda", dtype=torch.float16):
        ids = w2v_model(x).logits.argmax(-1)
    return w2v_processor.batch_decode(ids)[0].lower()


clips["w2v_transcript"] = [w2v_transcribe(y) for y in tqdm(waves, desc="wav2vec2")]
clips["w2v_words"] = clips["w2v_transcript"].str.split().str.len()

del w2v_model
free_gpu()

preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/162 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/423 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-960h-lv60-self
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


wav2vec2:   0%|          | 0/985 [00:00<?, ?it/s]

## 4. Whisper encoder embeddings

Whisper's encoder processes audio in 30 second windows and outputs 50 frames per second. We average each layer over the frames that contain audio (not the padding) and also keep the standard deviation.

In [5]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
whisper = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3", torch_dtype=torch.float16).to(DEVICE).eval()
WINDOW = 30 * SR


@torch.no_grad()
def whisper_embedding(y):
    windows = [y[i:i + WINDOW] for i in range(0, len(y), WINDOW)]
    if len(windows) > 1 and len(windows[-1]) < SR:  # ignore a tiny leftover at the end
        windows = windows[:-1]
    feats = processor.feature_extractor(windows, sampling_rate=SR, return_tensors="pt").input_features
    hidden = whisper.model.encoder(feats.to(DEVICE, torch.float16), output_hidden_states=True).hidden_states
    n_frames = [min(1500, math.ceil(len(w) / SR * 50)) for w in windows]
    layers = []
    for h in hidden:
        frames = torch.cat([h[k, :n].float() for k, n in enumerate(n_frames)])
        layers.append(torch.cat([frames.mean(0), frames.std(0)]))
    return torch.stack(layers).cpu().numpy()


emb_whisper = np.stack([whisper_embedding(y) for y in tqdm(waves, desc="Whisper encoder")])
print(emb_whisper.shape, "finite:", np.isfinite(emb_whisper).all())

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Whisper encoder:   0%|          | 0/985 [00:00<?, ?it/s]

(985, 33, 2560) finite: True


## 5. Whisper transcripts

Whisper tends to clean up what people say. To keep the speaker's mistakes, we give it a prompt that is itself disfluent and ungrammatical. This works for most clips, but on a few Whisper copies the prompt into the output or gets stuck in a loop. We detect those (using the wav2vec2 word count as a reference) and transcribe them again without the prompt.

In [6]:
PROMPT = "Um, so I, I goes to the, uh, market yesterday and, and I buy some vegetable."
prompt_ids = torch.tensor(processor.get_prompt_ids(PROMPT)).to(DEVICE)


@torch.no_grad()
def transcribe(y, use_prompt=True, **extra):
    long = len(y) > WINDOW
    inputs = processor(y, sampling_rate=SR, return_tensors="pt", return_attention_mask=True,
                       **(dict(truncation=False, padding="longest") if long else {})).to(DEVICE)
    inputs["input_features"] = inputs["input_features"].half()
    args = dict(language="en", task="transcribe", condition_on_prev_tokens=False, return_timestamps=long,
                temperature=(0.0, 0.2, 0.4), logprob_threshold=-1.0, compression_ratio_threshold=1.35)
    if use_prompt:
        args["prompt_ids"] = prompt_ids
    else:
        args["no_speech_threshold"] = 0.6
    text = processor.batch_decode(whisper.generate(**inputs, **args, **extra), skip_special_tokens=True)[0].strip()
    return text[len(PROMPT):].strip() if text.startswith(PROMPT) else text


def looks_broken(text, w2v_words):
    words = re.findall(r"[a-z']+", text.lower())
    top_5gram = max(Counter(zip(*[words[i:] for i in range(5)])).values(), default=0)
    n = len(text.split())
    return ("buy some vegetable" in text.lower() or top_5gram >= 5
            or n > 1.6 * max(w2v_words, 30) or (w2v_words > 30 and n < 0.6 * w2v_words))


if TRANSCRIPTS_CSV and os.path.exists(TRANSCRIPTS_CSV):
    old = pd.read_csv(TRANSCRIPTS_CSV)[["split", "filename", "whisper_transcript"]]
    clips = clips.merge(old, on=["split", "filename"], how="left")
    print("reusing transcripts from", TRANSCRIPTS_CSV)
else:
    clips["whisper_transcript"] = [transcribe(y) for y in tqdm(waves, desc="Whisper")]
    broken = [i for i in range(len(clips)) if scored[i] and looks_broken(clips.at[i, "whisper_transcript"], clips.at[i, "w2v_words"])]
    for i in tqdm(broken, desc="re-transcribing"):
        text = transcribe(waves[i], use_prompt=False)
        if looks_broken(text, clips.at[i, "w2v_words"]):
            text = transcribe(waves[i], use_prompt=False, no_repeat_ngram_size=6)
        clips.at[i, "whisper_transcript"] = text
    print("re-transcribed", len(broken), "clips")

clips["whisper_transcript"] = clips["whisper_transcript"].fillna("")
del whisper
free_gpu()

reusing transcripts from /kaggle/input/datasets/codewiz19/output-f/clips.csv


## 6. WavLM embeddings

In [7]:
from transformers import WavLMModel

wavlm = WavLMModel.from_pretrained("microsoft/wavlm-large").to(DEVICE).eval()


@torch.no_grad()
def wavlm_embedding(y):
    x = torch.tensor((y - y.mean()) / (y.std() + 1e-7))[None].to(DEVICE)
    with torch.autocast("cuda", dtype=torch.float16):
        hidden = wavlm(x, output_hidden_states=True).hidden_states
    return torch.stack([torch.cat([h[0].float().mean(0), h[0].float().std(0)]) for h in hidden]).cpu().numpy()


emb_wavlm = np.stack([wavlm_embedding(y) for y in tqdm(waves, desc="WavLM")])
print(emb_wavlm.shape, "finite:", np.isfinite(emb_wavlm).all())

del wavlm
free_gpu()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

WavLM:   0%|          | 0/985 [00:00<?, ?it/s]

(985, 25, 2048) finite: True


## 7. Clean the transcripts

The raters score grammar only, so hesitation should not count against the speaker. We remove fillers ("um", "uh") and exact repeats ("of the, of the"). A self-correction like "he go, he goes" is kept because it is not an exact repeat.

In [16]:
FILLERS = re.compile(r"\b(?:u+m+|u+h+|e+r+m*|a+h+|h+m+|m+h*m+)\b[,.]?\s*", re.I)
REPEATS = re.compile(r"\b(\w+(?:[ ,]+\w+){0,3})(?:,?\s+\1\b)+", re.I)


def clean(text):
    text = re.sub(r"\s+", " ", FILLERS.sub("", text)).strip()
    text = REPEATS.sub(r"\1", text)
    return re.sub(r"(^|[.?!]\s+)([a-z])", lambda m: m.group(1) + m.group(2).upper(), text)


def words_of(text):
    return re.findall(r"[a-z']+", text.lower())


clips["clean_transcript"] = clips["whisper_transcript"].map(clean)

# a few transcripts have almost no punctuation, so sentence-level features would be meaningless for them
n_words = clips["whisper_transcript"].str.split().str.len().clip(lower=1)
clips["unpunctuated"] = clips["whisper_transcript"].str.count(r"[.?!]") / n_words < 0.02

print(clips["whisper_transcript"].iloc[0][:200])
print(clips["clean_transcript"].iloc[0][:200])

Um, well, this, the playground is very, very beautiful and fun. There are, there are play tools like the merry-go-round, the swing. The students usually enjoy the swing. It has to be like, um, it's mo
Well, this, the playground is very beautiful and fun. There are play tools like the merry-go-round, the swing. The students usually enjoy the swing. It has to be like, it's more like a game of fair. B


## 8. Handcrafted features

- **Disfluency**: word count, filler rate, repetition rate (from the raw transcript)
- **Syntax** (spaCy): sentence length, parse depth, clauses, fragments, tenses, part-of-speech mix
- **Grammar errors** (LanguageTool): rule violations per 100 words, ignoring spelling and punctuation
- **Surprisal** (GPT-2) and **acceptability** (a RoBERTa model trained on the CoLA grammaticality dataset)
- **wav2vec2 transcript**: share of non-words and how much it disagrees with Whisper

In [17]:
def disfluency(raw):
    words = words_of(raw)
    no_fillers = words_of(FILLERS.sub("", raw))
    repeats = sum(no_fillers[i:i + k] == no_fillers[i + k:i + 2 * k] for k in (1, 2, 3) for i in range(len(no_fillers) - 2 * k + 1))
    return {"n_words": len(words),
            "filler_rate": len(FILLERS.findall(raw)) / max(len(words), 1),
            "repetition_rate": repeats / max(len(no_fillers), 1)}


disfl = pd.DataFrame([disfluency(t) for t in clips["whisper_transcript"]])
timing["speech_rate_wps"] = disfl["n_words"] / timing["speech_s"].clip(lower=1)

In [18]:
import spacy

if not spacy.util.is_package("en_core_web_sm"):
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])
nlp = spacy.load("en_core_web_sm")

CLAUSES = {"advcl", "ccomp", "xcomp", "acl", "relcl", "csubj"}
SUBJECTS = {"nsubj", "nsubjpass", "csubj", "expl"}
POS = ["NOUN", "VERB", "ADJ", "ADV", "ADP", "DET", "PRON", "CCONJ", "SCONJ", "AUX"]
SENTENCE_COLS = ["n_sentences", "mean_sent_len", "std_sent_len", "max_sent_len", "mean_tree_depth",
                 "max_tree_depth", "clauses_per_sent", "fragment_ratio", "passive_per_sent"]


def depth(token):
    return 1 + max((depth(c) for c in token.children), default=0)


def mattr(words, window=50):
    # type-token ratio over a moving window, so long answers are not penalised
    if len(words) < window:
        return len(set(words)) / max(len(words), 1)
    return np.mean([len(set(words[i:i + window])) / window for i in range(len(words) - window + 1)])


def syntax(doc):
    tokens = [t for t in doc if not (t.is_punct or t.is_space)]
    sents = [s for s in doc.sents if any(not t.is_punct for t in s)]
    n_tok, n_sent = max(len(tokens), 1), max(len(sents), 1)
    lengths = [sum(not t.is_punct for t in s) for s in sents] or [0]
    depths = [depth(s.root) for s in sents] or [0]
    fragments = sum(not (any(t.pos_ in ("VERB", "AUX") for t in s) and any(t.dep_ in SUBJECTS for t in s)) for s in sents)
    dists = [abs(t.i - t.head.i) for t in tokens if t.head.i != t.i]
    pos = Counter(t.pos_ for t in tokens)
    return {
        "n_sentences": len(sents),
        "mean_sent_len": np.mean(lengths),
        "std_sent_len": np.std(lengths),
        "max_sent_len": max(lengths),
        "mean_tree_depth": np.mean(depths),
        "max_tree_depth": max(depths),
        "clauses_per_sent": sum(t.dep_ in CLAUSES for t in doc) / n_sent,
        "fragment_ratio": fragments / n_sent,
        "mean_dep_distance": np.mean(dists) if dists else 0.0,
        "n_tenses": len({m for t in doc for m in t.morph.get("Tense")}),
        "n_verb_forms": len({m for t in doc for m in t.morph.get("VerbForm")}),
        "passive_per_sent": sum(t.dep_ in ("nsubjpass", "auxpass") for t in doc) / n_sent,
        "mattr": mattr([t.lower_ for t in tokens]),
        "mean_word_len": np.mean([len(t) for t in tokens]) if tokens else 0.0,
        **{f"pos_{p.lower()}": pos[p] / n_tok for p in POS},
    }


docs = list(nlp.pipe(clips["clean_transcript"], batch_size=64))
syn = pd.DataFrame([syntax(d) for d in docs])
syn.loc[clips["unpunctuated"], SENTENCE_COLS] = np.nan

In [11]:
# LanguageTool needs Java
if shutil.which("java") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y default-jre-headless > /dev/null", shell=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "language-tool-python"])
import language_tool_python

tool = language_tool_python.LanguageTool("en-US")
IGNORE = {"TYPOS", "TYPOGRAPHY", "CASING", "PUNCTUATION"}


def grammar_errors(text):
    n = max(len(words_of(text)), 1)
    cats = [m.category for m in tool.check(text)] if text else []
    return {"lt_grammar_per100": 100 * cats.count("GRAMMAR") / n,
            "lt_errors_per100": 100 * sum(c not in IGNORE for c in cats) / n}


lt = pd.DataFrame([grammar_errors(t) for t in tqdm(clips["clean_transcript"], desc="LanguageTool")])
tool.close()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 1.5 MB/s eta 0:00:00


LanguageTool:   0%|          | 0/985 [00:00<?, ?it/s]

In [12]:
from transformers import AutoModel, AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE).eval()


@torch.no_grad()
def surprisal(text):
    ids = gpt2_tok(gpt2_tok.bos_token + text, return_tensors="pt", truncation=True, max_length=1024).input_ids.to(DEVICE)
    if ids.shape[1] < 3:
        return {"lm_mean_nll": np.nan, "lm_p90_nll": np.nan}
    nll = torch.nn.functional.cross_entropy(gpt2(ids).logits[0, :-1], ids[0, 1:], reduction="none").cpu().numpy()
    return {"lm_mean_nll": nll.mean(), "lm_p90_nll": np.percentile(nll, 90)}


cola_tok = AutoTokenizer.from_pretrained("textattack/roberta-base-CoLA")
cola = AutoModelForSequenceClassification.from_pretrained("textattack/roberta-base-CoLA").to(DEVICE).eval()


@torch.no_grad()
def acceptability(doc):
    sents = [s.text for s in doc.sents if len(s) >= 3] or [doc.text or "."]
    enc = cola_tok(sents, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
    p = cola(**enc).logits.softmax(-1)[:, 1].cpu().numpy()  # probability the sentence is acceptable
    return {"cola_mean": p.mean(), "cola_min": p.min(), "cola_frac_bad": (p < 0.5).mean()}


lm = pd.DataFrame([surprisal(t) for t in tqdm(clips["clean_transcript"], desc="GPT-2")])
acc = pd.DataFrame([acceptability(d) for d in tqdm(docs, desc="CoLA")])
acc.loc[clips["unpunctuated"]] = np.nan

del gpt2, cola
free_gpu()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: textattack/roberta-base-CoLA
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

GPT-2:   0%|          | 0/985 [00:00<?, ?it/s]

CoLA:   0%|          | 0/985 [00:00<?, ?it/s]

In [13]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wordfreq"])
from wordfreq import zipf_frequency


def w2v_features(w2v_text, whisper_text):
    words = words_of(w2v_text)
    if not words:
        return {}
    return {
        "w2v_n_words": len(words),
        "w2v_unknown_rate": np.mean([zipf_frequency(w, "en") < 2 for w in words]),
        "w2v_repeat_rate": sum(a == b for a, b in zip(words, words[1:])) / len(words),
        "w2v_mean_word_len": np.mean([len(w) for w in words]),
        "asr_disagreement": 1 - difflib.SequenceMatcher(None, words, words_of(whisper_text), autojunk=False).ratio(),
    }


w2v = pd.DataFrame([w2v_features(a, b) for a, b in zip(clips["w2v_transcript"], clips["clean_transcript"])])
features = pd.concat([disfl, syn, lt, lm, acc, w2v, timing.drop(columns="speech_s")], axis=1).astype(float)

# quick check: correlation of each feature with the label
train_rows = scored & clips["split"].eq("train").to_numpy()
print(features[train_rows].corrwith(clips.loc[train_rows, "label"], method="spearman").sort_values().round(2).to_string())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 9.7 MB/s eta 0:00:00
cola_frac_bad         -0.46
asr_disagreement      -0.42
w2v_unknown_rate      -0.37
lm_mean_nll           -0.35
repetition_rate       -0.30
lm_p90_nll            -0.25
lt_grammar_per100     -0.23
lt_errors_per100      -0.22
w2v_repeat_rate       -0.22
pos_det               -0.08
pos_pron              -0.07
pauses_per_min        -0.06
pos_cconj             -0.06
long_pauses_per_min   -0.05
mean_pause_s          -0.03
mean_dep_distance     -0.03
pos_noun              -0.02
filler_rate           -0.02
mean_run_s             0.00
fragment_ratio         0.00
pos_aux                0.01
pos_sconj              0.03
n_tenses               0.04
pos_adp                0.04
std_sent_len           0.05
mean_sent_len          0.06
passive_per_sent       0.07
speech_

## 9. Text embeddings

Mean of every layer over the tokens of the clean transcript. For Qwen we skip the first token: decoder LLMs put very large values there, which would dominate the average.

In [14]:
@torch.no_grad()
def text_embedding(model_id, texts, dtype=torch.float32, skip_first=False):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id, torch_dtype=dtype).to(DEVICE).eval()
    out = []
    for text in tqdm(texts, desc=model_id):
        enc = tok(text or ".", return_tensors="pt", truncation=True, max_length=1024 if skip_first else 512).to(DEVICE)
        hidden = model(**enc, output_hidden_states=True).hidden_states
        start = 1 if skip_first and enc["input_ids"].shape[1] > 1 else 0
        out.append(torch.stack([h[0, start:].float().mean(0) for h in hidden]).cpu().numpy())
    del model
    free_gpu()
    return np.stack(out)


emb_roberta = text_embedding("roberta-large", clips["clean_transcript"])
emb_qwen = text_embedding("Qwen/Qwen2.5-1.5B", clips["clean_transcript"], dtype=torch.float16, skip_first=True)
print(emb_roberta.shape, emb_qwen.shape)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen/Qwen2.5-1.5B:   0%|          | 0/985 [00:00<?, ?it/s]

(985, 25, 1024) (985, 29, 1536)


## 10. Save

In [15]:
cols = ["split", "filename", "label", "duration_s", "dyn_range_db", "whisper_transcript", "clean_transcript", "w2v_transcript", "unpunctuated"]
clips[cols].to_csv(f"{OUT_DIR}/clips.csv", index=False)
features.to_csv(f"{OUT_DIR}/features.csv", index=False)

for name, emb in [("wavlm", emb_wavlm), ("whisper", emb_whisper), ("roberta", emb_roberta), ("qwen", emb_qwen)]:
    assert len(emb) == len(clips) and np.isfinite(emb).all(), name
    np.save(f"{OUT_DIR}/emb_{name}.npy", emb.astype(np.float32))

print(os.listdir(OUT_DIR))

['emb_wavlm.npy', '.virtual_documents', 'features.csv', 'emb_qwen.npy', 'emb_whisper.npy', 'clips.csv', 'emb_roberta.npy']
